# FocusFlow Agent — Milestone 3 Google Calendar Integration Notebook

This notebook extends the Milestone 1 + 2 FocusFlow workflow with **Google Calendar integration**.

By the end, FocusFlow can:

1. Turn a messy task dump into a prioritized plan
2. Generate a Slack-ready daily summary
3. Prepare Google Calendar event payloads
4. Optionally create real Google Calendar holds

The notebook is safe by default: Slack posting and Google Calendar creation are both disabled unless you explicitly turn them on.

## 1. Setup

Run this cell first. It installs the small set of packages needed for Milestone 1.

- `openai`: optional, only needed when you turn off mock mode and call a real LLM
- `pandas`: displays task and schedule tables cleanly


In [1]:
# Install dependencies.
# In Colab, this usually takes a few seconds.
!pip install openai pandas requests -q

## 2. Imports and configuration

This notebook is designed to be workshop-safe.

By default, `USE_MOCK_MODE = True`, so the notebook runs without any API key. This makes it easy to test the full flow in a live room.

When you are ready to call a real model, set:

```python
USE_MOCK_MODE = False
```

Then enter your OpenAI API key when prompted.


In [2]:
import os
import json
import re
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from getpass import getpass

import pandas as pd
from IPython.display import display, Markdown

# Workshop-safe default.
# Keep this True while teaching the notebook flow.
# Set to False when you want to call a real OpenAI model.
USE_MOCK_MODE = True

# You can change this to any model available in your OpenAI account.
MODEL_NAME = "gpt-4.1-mini"

TIMEZONE = "America/Los_Angeles"
TODAY_DATE = datetime.now(ZoneInfo(TIMEZONE)).date().isoformat()

print(f"FocusFlow notebook loaded. Today: {TODAY_DATE}. Timezone: {TIMEZONE}.")
print(f"Mock mode: {USE_MOCK_MODE}")

FocusFlow notebook loaded. Today: 2026-05-21. Timezone: America/Los_Angeles.
Mock mode: True


## 3. Optional: API key setup

Only run this cell if you set `USE_MOCK_MODE = False`.

For the first workshop run, you can leave mock mode on and skip this cell.


In [3]:
if not USE_MOCK_MODE:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    print("API key configured.")
else:
    print("Mock mode is ON. No API key needed for Milestone 1 demo.")

Mock mode is ON. No API key needed for Milestone 1 demo.


## 4. Sample messy task dump

This is the input FocusFlow will organize.

For the workshop, start with this sample so everyone sees the same output first. Later, attendees can replace it with their own task dump.


In [4]:
task_dump = """
I need to prepare slides for Friday, email Sam, review an AI paper,
book a dentist appointment, finish project update, go to the gym,
pay rent, and plan the meetup agenda.

I have 4 hours today and prefer deep work in the morning.
""".strip()

print(task_dump)

I need to prepare slides for Friday, email Sam, review an AI paper,
book a dentist appointment, finish project update, go to the gym,
pay rent, and plan the meetup agenda.

I have 4 hours today and prefer deep work in the morning.


## 5. FocusFlow agent workflow

FocusFlow is more than a prompt that rewrites a to-do list.

It follows a simple agentic planning loop:

```text
Extract → Categorize → Prioritize → Schedule → Validate → Preview Integrations
```

This is the core teaching point of Milestone 1.


## 6. Define the expected output format

A high-quality agent should return predictable structured output.

This schema gives us a contract between the AI planning step and the later integrations.


In [5]:
EXPECTED_OUTPUT_FORMAT = {
    "tasks": [
        {
            "task": "string",
            "category": "Work | Personal | Admin | Health | Learning | Community | Other",
            "priority": "High | Medium | Low",
            "effort": "High | Medium | Low",
            "urgency": "High | Medium | Low",
            "when": "Today | This Week | Later",
            "estimated_minutes": 30,
            "reason": "short explanation"
        }
    ],
    "schedule": [
        {
            "title": "string",
            "start_time": "ISO 8601 datetime",
            "end_time": "ISO 8601 datetime",
            "type": "deep_work | admin | personal | break | flexible",
            "reason": "short explanation"
        }
    ],
    "risks": ["string"],
    "next_best_action": "string",
    "slack_message": "string",
    "calendar_preview": [
        {
            "summary": "string",
            "start": "ISO 8601 datetime",
            "end": "ISO 8601 datetime"
        }
    ]
}

EXPECTED_OUTPUT_FORMAT

{'tasks': [{'task': 'string',
   'category': 'Work | Personal | Admin | Health | Learning | Community | Other',
   'priority': 'High | Medium | Low',
   'effort': 'High | Medium | Low',
   'urgency': 'High | Medium | Low',
   'when': 'Today | This Week | Later',
   'estimated_minutes': 30,
   'reason': 'short explanation'}],
 'schedule': [{'title': 'string',
   'start_time': 'ISO 8601 datetime',
   'end_time': 'ISO 8601 datetime',
   'type': 'deep_work | admin | personal | break | flexible',
   'reason': 'short explanation'}],
 'risks': ['string'],
 'next_best_action': 'string',
 'slack_message': 'string',
 'calendar_preview': [{'summary': 'string',
   'start': 'ISO 8601 datetime',
   'end': 'ISO 8601 datetime'}]}

## 7. Define the FocusFlow prompt

This is the core agent instruction.

The prompt tells FocusFlow exactly how to reason about the messy task dump and exactly what format to return.


In [6]:
FOCUSFLOW_SYSTEM_PROMPT = """
You are FocusFlow, a personal planning AI agent.

Your goal is to turn a messy task dump into a realistic daily plan.

You must follow this workflow:
1. Extract individual tasks from the messy input.
2. Categorize each task.
3. Estimate urgency, effort, and importance.
4. Prioritize tasks into Today, This Week, and Later.
5. Create a realistic schedule for today based on the user's available time and preferences.
6. Flag risks such as overload, missing deadlines, unclear tasks, or too many deep-work items.
7. Recommend the single next best action.
8. Generate a Slack-ready daily plan message.
9. Generate calendar-ready event previews.

Planning rules:
- Do not schedule more work than the available time allows.
- Prefer deep work in the morning if the user requests it.
- Batch small admin tasks together.
- Include breaks between deep-work blocks.
- If a task has an explicit deadline, treat it as more urgent.
- If a task has no deadline, infer priority conservatively.
- If the plan is overloaded, move lower-priority tasks to This Week or Later.
- Be practical, concise, and realistic.


Slack message formatting rules:
- Format slack_message for Slack using simple mrkdwn.
- Use *bold* section headers.
- Use numbered priorities and bullet schedule items.
- Keep the message concise and readable.
- Include risks and the next best action.

Return valid JSON only.
""".strip()


def build_user_prompt(task_dump, today_date, timezone):
    """Build the user prompt passed to the LLM."""
    return f"""
Task dump:
{task_dump}

Today's date:
{today_date}

Timezone:
{timezone}

Return JSON with exactly these keys:
- tasks
- schedule
- risks
- next_best_action
- slack_message
- calendar_preview

Each task must include:
- task
- category
- priority
- effort
- urgency
- when
- estimated_minutes
- reason

Each schedule item must include:
- title
- start_time
- end_time
- type
- reason

Each calendar_preview item must include:
- summary
- start
- end
""".strip()

print(FOCUSFLOW_SYSTEM_PROMPT[:500] + "...")

You are FocusFlow, a personal planning AI agent.

Your goal is to turn a messy task dump into a realistic daily plan.

You must follow this workflow:
1. Extract individual tasks from the messy input.
2. Categorize each task.
3. Estimate urgency, effort, and importance.
4. Prioritize tasks into Today, This Week, and Later.
5. Create a realistic schedule for today based on the user's available time and preferences.
6. Flag risks such as overload, missing deadlines, unclear tasks, or too many deep-...


## 8. Helper functions

These helpers make the notebook robust:

- `extract_json`: handles model output that accidentally includes markdown fences
- `validate_plan`: checks that the plan has the required structure
- `make_mock_plan`: gives us a reliable demo output without calling an API


In [7]:
def extract_json(raw_text):
    """Extract JSON from a model response.

    Some models return JSON wrapped in ```json fences. This function strips
    those fences and parses the JSON safely.
    """
    text = raw_text.strip()

    # Remove common markdown code fences if present.
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)


def validate_plan(plan, verbose=True):
    """Validate the core FocusFlow plan structure before display or integration."""
    required_keys = [
        "tasks",
        "schedule",
        "risks",
        "next_best_action",
        "slack_message",
        "calendar_preview",
    ]

    missing = [key for key in required_keys if key not in plan]
    if missing:
        if verbose:
            print("Missing required keys:", missing)
        return False

    if not isinstance(plan["tasks"], list):
        if verbose:
            print("Expected 'tasks' to be a list.")
        return False

    if not isinstance(plan["schedule"], list):
        if verbose:
            print("Expected 'schedule' to be a list.")
        return False

    if not isinstance(plan["risks"], list):
        if verbose:
            print("Expected 'risks' to be a list.")
        return False

    if not isinstance(plan["calendar_preview"], list):
        if verbose:
            print("Expected 'calendar_preview' to be a list.")
        return False

    if verbose:
        print("Plan structure looks good.")
    return True


def iso_at(today_date, hour, minute=0, timezone=TIMEZONE):
    """Create an ISO 8601 datetime string for a given date/time/timezone."""
    tz = ZoneInfo(timezone)
    y, m, d = map(int, today_date.split("-"))
    dt = datetime(y, m, d, hour, minute, tzinfo=tz)
    return dt.isoformat()


def make_mock_plan(today_date=TODAY_DATE, timezone=TIMEZONE):
    """Return a deterministic example plan for demo and testing.

    This lets the workshop run smoothly even without API keys.
    """
    return {
        "tasks": [
            {
                "task": "Finish project update",
                "category": "Work",
                "priority": "High",
                "effort": "High",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 90,
                "reason": "Important work task and likely needed before other updates."
            },
            {
                "task": "Prepare slides for Friday",
                "category": "Work",
                "priority": "High",
                "effort": "High",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 75,
                "reason": "Explicit deadline makes this time-sensitive."
            },
            {
                "task": "Plan the meetup agenda",
                "category": "Community",
                "priority": "High",
                "effort": "Medium",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 45,
                "reason": "Useful to make progress while planning context is fresh."
            },
            {
                "task": "Pay rent",
                "category": "Admin",
                "priority": "High",
                "effort": "Low",
                "urgency": "High",
                "when": "Today",
                "estimated_minutes": 10,
                "reason": "Quick task with potentially high consequence if delayed."
            },
            {
                "task": "Email Sam",
                "category": "Admin",
                "priority": "Medium",
                "effort": "Low",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 15,
                "reason": "Quick communication task that can be batched with admin work."
            },
            {
                "task": "Go to the gym",
                "category": "Health",
                "priority": "Medium",
                "effort": "Medium",
                "urgency": "Medium",
                "when": "Today",
                "estimated_minutes": 60,
                "reason": "Health task fits better outside the 4-hour work planning window."
            },
            {
                "task": "Review an AI paper",
                "category": "Learning",
                "priority": "Medium",
                "effort": "High",
                "urgency": "Low",
                "when": "This Week",
                "estimated_minutes": 60,
                "reason": "Valuable but not as urgent as deadline-driven work."
            },
            {
                "task": "Book dentist appointment",
                "category": "Personal",
                "priority": "Low",
                "effort": "Low",
                "urgency": "Low",
                "when": "This Week",
                "estimated_minutes": 10,
                "reason": "Quick personal admin task that can be done later this week."
            },
        ],
        "schedule": [
            {
                "title": "Finish project update",
                "start_time": iso_at(today_date, 9, 0, timezone),
                "end_time": iso_at(today_date, 10, 30, timezone),
                "type": "deep_work",
                "reason": "Use morning focus for the highest-priority deep-work task."
            },
            {
                "title": "Break",
                "start_time": iso_at(today_date, 10, 30, timezone),
                "end_time": iso_at(today_date, 10, 45, timezone),
                "type": "break",
                "reason": "Short reset between deep-work blocks."
            },
            {
                "title": "Prepare slide outline",
                "start_time": iso_at(today_date, 10, 45, timezone),
                "end_time": iso_at(today_date, 12, 0, timezone),
                "type": "deep_work",
                "reason": "Deadline-driven work benefits from protected focus time."
            },
            {
                "title": "Admin batch: pay rent + email Sam",
                "start_time": iso_at(today_date, 14, 0, timezone),
                "end_time": iso_at(today_date, 14, 30, timezone),
                "type": "admin",
                "reason": "Batch low-effort admin tasks together."
            },
            {
                "title": "Plan meetup agenda",
                "start_time": iso_at(today_date, 14, 30, timezone),
                "end_time": iso_at(today_date, 15, 15, timezone),
                "type": "flexible",
                "reason": "Medium-effort planning task fits after urgent work is complete."
            },
        ],
        "risks": [
            "The full task list exceeds the 4-hour planning window, so lower-priority work should move to later this week.",
            "Reviewing the AI paper is a deep-work task and should not be squeezed into an already full day.",
            "Prepare slides needs a clear definition of done, such as outline only vs. full deck."
        ],
        "next_best_action": "Start with the project update before opening email or doing smaller admin tasks.",
        "slack_message": """*FocusFlow Daily Plan*\n\n*Top priorities*\n1. Finish project update\n2. Prepare slide outline\n3. Pay rent + email Sam\n\n*Schedule*\n• 9:00–10:30 — Finish project update\n• 10:45–12:00 — Prepare slide outline\n• 2:00–2:30 — Admin batch\n• 2:30–3:15 — Plan meetup agenda\n\n*Risk*\nYour list is larger than today's 4-hour planning window. Move paper review and dentist booking to later this week.\n\n*Next best action*\nStart with the project update.""",
        "calendar_preview": [
            {
                "summary": "FocusFlow: Finish project update",
                "start": iso_at(today_date, 9, 0, timezone),
                "end": iso_at(today_date, 10, 30, timezone)
            },
            {
                "summary": "FocusFlow: Prepare slide outline",
                "start": iso_at(today_date, 10, 45, timezone),
                "end": iso_at(today_date, 12, 0, timezone)
            },
            {
                "summary": "FocusFlow: Admin batch: pay rent + email Sam",
                "start": iso_at(today_date, 14, 0, timezone),
                "end": iso_at(today_date, 14, 30, timezone)
            },
            {
                "summary": "FocusFlow: Plan meetup agenda",
                "start": iso_at(today_date, 14, 30, timezone),
                "end": iso_at(today_date, 15, 15, timezone)
            }
        ]
    }


## 9. Generate the FocusFlow plan

This function supports two modes:

- **Mock mode:** deterministic demo output, no API key required
- **Real mode:** calls the OpenAI Responses API and parses JSON output

For a live workshop, start with mock mode. Then show how to turn real mode on.


In [8]:
def generate_focusflow_plan(task_dump, today_date=TODAY_DATE, timezone=TIMEZONE, use_mock=USE_MOCK_MODE):
    """Generate a FocusFlow plan from a messy task dump."""
    if use_mock:
        return make_mock_plan(today_date=today_date, timezone=timezone)

    # Import OpenAI only when real mode is used, so mock mode stays lightweight.
    from openai import OpenAI

    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

    response = client.responses.create(
        model=MODEL_NAME,
        input=[
            {"role": "system", "content": FOCUSFLOW_SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(task_dump, today_date, timezone)},
        ],
        temperature=0.2,
    )

    return extract_json(response.output_text)


plan = generate_focusflow_plan(task_dump)
validate_plan(plan)

# Show raw JSON preview.
print(json.dumps(plan, indent=2)[:2000] + "\n...")

Plan structure looks good.
{
  "tasks": [
    {
      "task": "Finish project update",
      "category": "Work",
      "priority": "High",
      "effort": "High",
      "urgency": "High",
      "when": "Today",
      "estimated_minutes": 90,
      "reason": "Important work task and likely needed before other updates."
    },
    {
      "task": "Prepare slides for Friday",
      "category": "Work",
      "priority": "High",
      "effort": "High",
      "urgency": "High",
      "when": "Today",
      "estimated_minutes": 75,
      "reason": "Explicit deadline makes this time-sensitive."
    },
    {
      "task": "Plan the meetup agenda",
      "category": "Community",
      "priority": "High",
      "effort": "Medium",
      "urgency": "Medium",
      "when": "Today",
      "estimated_minutes": 45,
      "reason": "Useful to make progress while planning context is fresh."
    },
    {
      "task": "Pay rent",
      "category": "Admin",
      "priority": "High",
      "effort": "Low",

## 10. Prioritized task table

This table is the first high-value output.

It shows how FocusFlow turned messy text into structured tasks with categories, priorities, urgency, effort, and reasoning.


In [9]:
tasks_df = pd.DataFrame(plan["tasks"])

task_columns = [
    "task",
    "category",
    "priority",
    "effort",
    "urgency",
    "when",
    "estimated_minutes",
    "reason",
]

tasks_df = tasks_df[task_columns]
display(tasks_df)

,task,category,priority,effort,urgency,when,estimated_minutes,reason
0,Finish project update,Work,High,High,High,Today,90,Important work task and likely needed before o...
1,Prepare slides for Friday,Work,High,High,High,Today,75,Explicit deadline makes this time-sensitive.
2,Plan the meetup agenda,Community,High,Medium,Medium,Today,45,Useful to make progress while planning context...
3,Pay rent,Admin,High,Low,High,Today,10,Quick task with potentially high consequence i...
4,Email Sam,Admin,Medium,Low,Medium,Today,15,Quick communication task that can be batched w...
5,Go to the gym,Health,Medium,Medium,Medium,Today,60,Health task fits better outside the 4-hour wor...
6,Review an AI paper,Learning,Medium,High,Low,This Week,60,Valuable but not as urgent as deadline-driven ...
7,Book dentist appointment,Personal,Low,Low,Low,This Week,10,Quick personal admin task that can be done lat...


## 11. Today’s schedule

This is the second high-value output.

The agent does not just list tasks. It turns selected tasks into a realistic schedule.


In [10]:
schedule_df = pd.DataFrame(plan["schedule"])

schedule_columns = [
    "title",
    "start_time",
    "end_time",
    "type",
    "reason",
]

schedule_df = schedule_df[schedule_columns]
display(schedule_df)

,title,start_time,end_time,type,reason
0,Finish project update,2026-05-21T09:00:00-07:00,2026-05-21T10:30:00-07:00,deep_work,Use morning focus for the highest-priority dee...
1,Break,2026-05-21T10:30:00-07:00,2026-05-21T10:45:00-07:00,break,Short reset between deep-work blocks.
2,Prepare slide outline,2026-05-21T10:45:00-07:00,2026-05-21T12:00:00-07:00,deep_work,Deadline-driven work benefits from protected f...
3,Admin batch: pay rent + email Sam,2026-05-21T14:00:00-07:00,2026-05-21T14:30:00-07:00,admin,Batch low-effort admin tasks together.
4,Plan meetup agenda,2026-05-21T14:30:00-07:00,2026-05-21T15:15:00-07:00,flexible,Medium-effort planning task fits after urgent ...


## 12. Plan check: risks and next best action

This is the validation step.

A useful planning agent should notice when the plan is overloaded, unclear, or unrealistic.


In [11]:
display(Markdown("### Risks / Plan Check"))
for risk in plan["risks"]:
    display(Markdown(f"- {risk}"))

display(Markdown("### Next Best Action"))
display(Markdown(f"**{plan['next_best_action']}**"))

### Risks / Plan Check

- The full task list exceeds the 4-hour planning window, so lower-priority work should move to later this week.

- Reviewing the AI paper is a deep-work task and should not be squeezed into an already full day.

- Prepare slides needs a clear definition of done, such as outline only vs. full deck.

### Next Best Action

**Start with the project update before opening email or doing smaller admin tasks.**

## 13. Slack-message preview

Milestone 2 will send this message to Slack.

For Milestone 1, we only generate and preview it.


In [12]:
display(Markdown("### Slack Daily Plan Preview"))
display(Markdown(plan["slack_message"]))

### Slack Daily Plan Preview

*FocusFlow Daily Plan*

*Top priorities*
1. Finish project update
2. Prepare slide outline
3. Pay rent + email Sam

*Schedule*
• 9:00–10:30 — Finish project update
• 10:45–12:00 — Prepare slide outline
• 2:00–2:30 — Admin batch
• 2:30–3:15 — Plan meetup agenda

*Risk*
Your list is larger than today's 4-hour planning window. Move paper review and dentist booking to later this week.

*Next best action*
Start with the project update.

## 14. Calendar hold preview

Milestone 3 will turn these previews into real Google Calendar holds.

For Milestone 1, we only generate calendar-ready blocks.


In [13]:
calendar_df = pd.DataFrame(plan["calendar_preview"])
display(calendar_df)

,summary,start,end
0,FocusFlow: Finish project update,2026-05-21T09:00:00-07:00,2026-05-21T10:30:00-07:00
1,FocusFlow: Prepare slide outline,2026-05-21T10:45:00-07:00,2026-05-21T12:00:00-07:00
2,FocusFlow: Admin batch: pay rent + email Sam,2026-05-21T14:00:00-07:00,2026-05-21T14:30:00-07:00
3,FocusFlow: Plan meetup agenda,2026-05-21T14:30:00-07:00,2026-05-21T15:15:00-07:00


## 15. Calendar event payload preview

This is what Milestone 3 will send to the Google Calendar API.

For now, this cell only creates the payloads locally.


In [14]:
calendar_event_payloads = []

for item in plan["calendar_preview"]:
    calendar_event_payloads.append({
        "summary": item["summary"],
        "description": "Created by FocusFlow Agent",
        "start": {
            "dateTime": item["start"],
            "timeZone": TIMEZONE,
        },
        "end": {
            "dateTime": item["end"],
            "timeZone": TIMEZONE,
        },
    })

print(json.dumps(calendar_event_payloads, indent=2))

[
  {
    "summary": "FocusFlow: Finish project update",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-21T09:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-21T10:30:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summary": "FocusFlow: Prepare slide outline",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-21T10:45:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-21T12:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summary": "FocusFlow: Admin batch: pay rent + email Sam",
    "description": "Created by FocusFlow Agent",
    "start": {
      "dateTime": "2026-05-21T14:00:00-07:00",
      "timeZone": "America/Los_Angeles"
    },
    "end": {
      "dateTime": "2026-05-21T14:30:00-07:00",
      "timeZone": "America/Los_Angeles"
    }
  },
  {
    "summa

## 16. Try your own messy task dump

Replace the text below with your own task dump.

This is the part attendees should personalize during the workshop.


In [15]:
my_task_dump = """
I need to finish a project update, reply to two emails, prep for a meeting,
clean up my notes, go to the gym, buy groceries, and read one AI article.
I only have 3 hours today and I want to avoid doing deep work after 3 PM.
""".strip()

my_plan = generate_focusflow_plan(my_task_dump)
validate_plan(my_plan)

my_tasks_df = pd.DataFrame(my_plan["tasks"])[task_columns]
my_schedule_df = pd.DataFrame(my_plan["schedule"])[schedule_columns]

display(Markdown("### My Prioritized Tasks"))
display(my_tasks_df)

display(Markdown("### My Schedule"))
display(my_schedule_df)

display(Markdown("### My Next Best Action"))
display(Markdown(f"**{my_plan['next_best_action']}**"))

Plan structure looks good.


### My Prioritized Tasks

,task,category,priority,effort,urgency,when,estimated_minutes,reason
0,Finish project update,Work,High,High,High,Today,90,Important work task and likely needed before o...
1,Prepare slides for Friday,Work,High,High,High,Today,75,Explicit deadline makes this time-sensitive.
2,Plan the meetup agenda,Community,High,Medium,Medium,Today,45,Useful to make progress while planning context...
3,Pay rent,Admin,High,Low,High,Today,10,Quick task with potentially high consequence i...
4,Email Sam,Admin,Medium,Low,Medium,Today,15,Quick communication task that can be batched w...
5,Go to the gym,Health,Medium,Medium,Medium,Today,60,Health task fits better outside the 4-hour wor...
6,Review an AI paper,Learning,Medium,High,Low,This Week,60,Valuable but not as urgent as deadline-driven ...
7,Book dentist appointment,Personal,Low,Low,Low,This Week,10,Quick personal admin task that can be done lat...


### My Schedule

,title,start_time,end_time,type,reason
0,Finish project update,2026-05-21T09:00:00-07:00,2026-05-21T10:30:00-07:00,deep_work,Use morning focus for the highest-priority dee...
1,Break,2026-05-21T10:30:00-07:00,2026-05-21T10:45:00-07:00,break,Short reset between deep-work blocks.
2,Prepare slide outline,2026-05-21T10:45:00-07:00,2026-05-21T12:00:00-07:00,deep_work,Deadline-driven work benefits from protected f...
3,Admin batch: pay rent + email Sam,2026-05-21T14:00:00-07:00,2026-05-21T14:30:00-07:00,admin,Batch low-effort admin tasks together.
4,Plan meetup agenda,2026-05-21T14:30:00-07:00,2026-05-21T15:15:00-07:00,flexible,Medium-effort planning task fits after urgent ...


### My Next Best Action

**Start with the project update before opening email or doing smaller admin tasks.**

## 17. Optional: Export Milestone 1 outputs

This saves the tables as CSV files.

In Colab, you can download them from the file browser on the left.


In [16]:
tasks_df.to_csv("focusflow_tasks.csv", index=False)
schedule_df.to_csv("focusflow_schedule.csv", index=False)
calendar_df.to_csv("focusflow_calendar_preview.csv", index=False)

print("Exported:")
print("- focusflow_tasks.csv")
print("- focusflow_schedule.csv")
print("- focusflow_calendar_preview.csv")

Exported:
- focusflow_tasks.csv
- focusflow_schedule.csv
- focusflow_calendar_preview.csv


## 18. Milestone 1 complete

You now have **FocusFlow v1**:

```text
Messy task dump
    ↓
Structured planning agent
    ↓
Prioritized task table
    ↓
Today’s schedule
    ↓
Risk check + next best action
    ↓
Slack preview + calendar preview
```

Next milestones:

- **Milestone 2:** Send the Slack daily plan message to a real Slack channel
- **Milestone 3:** Create real Google Calendar holds from the calendar preview payloads


## 19. Milestone 2: Slack integration

Milestone 1 generated a Slack-ready daily planning message.

In Milestone 2, we add the first real external integration: posting that message to a Slack channel using a Slack incoming webhook.

**Important safety rule:** treat your Slack webhook URL like a password. Do not paste it into GitHub, screenshots, or public notebooks.

## 20. Slack integration configuration

For a safe workshop demo, Slack posting is disabled by default.

To post to Slack:

1. Create a Slack incoming webhook for your test channel.
2. Set `ENABLE_SLACK_POSTING = True`.
3. Run the cell and paste the webhook URL when prompted.

When this notebook is run without a webhook, it still validates the Slack message and skips real posting safely.

In [17]:
# Milestone 2 configuration.
# Keep this False for dry runs, GitHub commits, and workshop-safe execution.
# Set to True only when you are ready to post to a real Slack channel.
ENABLE_SLACK_POSTING = False

# Optional: enable a small webhook test message before posting the FocusFlow plan.
SEND_TEST_MESSAGE = False

# Optional: post the FocusFlow daily plan after the preview.
SEND_FOCUSFLOW_PLAN = False

# The notebook first checks for an environment variable.
# This avoids hardcoding secrets in notebook cells.
SLACK_WEBHOOK_URL = os.environ.get("SLACK_WEBHOOK_URL", "").strip()

if ENABLE_SLACK_POSTING and not SLACK_WEBHOOK_URL:
    SLACK_WEBHOOK_URL = getpass("Enter your Slack webhook URL: ").strip()

if ENABLE_SLACK_POSTING:
    print("Slack posting is ENABLED for this session.")
    print("Webhook loaded:", bool(SLACK_WEBHOOK_URL))
else:
    print("Slack posting is disabled. Running in safe preview mode.")

Slack posting is disabled. Running in safe preview mode.


## 21. Define the Slack posting helper

This helper sends a plain-text message to Slack using the incoming webhook URL.

The function is intentionally small so beginners can understand the integration:

```text
FocusFlow message → HTTP POST → Slack channel
```

In [18]:
def post_to_slack(message, webhook_url):
    """Post a message to Slack using an incoming webhook.

    Args:
        message (str): The Slack-ready message text.
        webhook_url (str): Slack incoming webhook URL.

    Returns:
        bool: True if Slack returns a successful response.

    Raises:
        ValueError: If the message or webhook URL is missing.
        RuntimeError: If Slack returns a non-200 response.
    """
    if not webhook_url:
        raise ValueError("Missing Slack webhook URL.")

    if not message or not message.strip():
        raise ValueError("Cannot post an empty Slack message.")

    response = requests.post(
        webhook_url,
        json={"text": message},
        timeout=10,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Slack post failed. Status: {response.status_code}. Response: {response.text}"
        )

    return True


print("Slack helper loaded.")

Slack helper loaded.


## 22. Validate the Slack message before posting

Before sending anything to Slack, check that FocusFlow produced a usable message.

This is a small but important reliability step. Real integrations should validate the payload before executing an external action.

In [19]:
def validate_slack_message(plan):
    """Validate that the FocusFlow plan contains a Slack-ready message."""
    if "slack_message" not in plan:
        raise KeyError("Plan is missing the 'slack_message' key.")

    message = plan["slack_message"]

    if not isinstance(message, str):
        raise TypeError("'slack_message' must be a string.")

    if not message.strip():
        raise ValueError("'slack_message' is empty.")

    print("Slack message validation passed.")
    print(f"Message length: {len(message)} characters")
    return message


focusflow_slack_message = validate_slack_message(plan)

Slack message validation passed.
Message length: 434 characters


## 23. Preview the exact Slack message

This is the message that will be posted in Milestone 2.

Previewing before posting protects you from sending bad, repeated, or overly verbose messages to your Slack channel.

In [20]:
display(Markdown("### Final Slack Message Preview"))
print(focusflow_slack_message)

### Final Slack Message Preview

*FocusFlow Daily Plan*

*Top priorities*
1. Finish project update
2. Prepare slide outline
3. Pay rent + email Sam

*Schedule*
• 9:00–10:30 — Finish project update
• 10:45–12:00 — Prepare slide outline
• 2:00–2:30 — Admin batch
• 2:30–3:15 — Plan meetup agenda

*Risk*
Your list is larger than today's 4-hour planning window. Move paper review and dentist booking to later this week.

*Next best action*
Start with the project update.


## 24. Optional: send a test message to Slack

Run this before sending the real FocusFlow plan if you want to confirm your webhook works.

By default, this cell does not send anything unless:

```python
ENABLE_SLACK_POSTING = True
SEND_TEST_MESSAGE = True
```

In [21]:
if ENABLE_SLACK_POSTING and SEND_TEST_MESSAGE:
    test_message = "Hello from FocusFlow Agent. Slack integration is working."
    post_to_slack(test_message, SLACK_WEBHOOK_URL)
    print("Test message posted to Slack.")
else:
    print("Skipping Slack test message. Set ENABLE_SLACK_POSTING=True and SEND_TEST_MESSAGE=True to send it.")

Skipping Slack test message. Set ENABLE_SLACK_POSTING=True and SEND_TEST_MESSAGE=True to send it.


## 25. Optional: post the FocusFlow daily plan to Slack

This is the real Milestone 2 action.

By default, this cell is safe and does not post. To send the message:

```python
ENABLE_SLACK_POSTING = True
SEND_FOCUSFLOW_PLAN = True
```

In [22]:
if ENABLE_SLACK_POSTING and SEND_FOCUSFLOW_PLAN:
    post_to_slack(focusflow_slack_message, SLACK_WEBHOOK_URL)
    print("FocusFlow daily plan posted to Slack.")
else:
    print("Preview only. FocusFlow daily plan was not posted to Slack.")

Preview only. FocusFlow daily plan was not posted to Slack.


## 26. Milestone 2 complete

You now have a notebook that can:

1. Generate a FocusFlow plan from a messy task dump
2. Display the prioritized task table
3. Display the daily schedule
4. Preview a Slack-ready daily plan message
5. Optionally post the message to a real Slack channel

Next milestone:

```text
Milestone 3: Create real Google Calendar holds from the schedule output.
```

## 27. Milestone 3: Google Calendar integration

In Milestone 1, FocusFlow generated a schedule and calendar-ready preview.

In Milestone 2, FocusFlow prepared and optionally posted a Slack daily plan.

In Milestone 3, we add **Google Calendar integration** so the generated schedule can become real calendar holds.

The pattern is:

```text
FocusFlow schedule → Calendar event payloads → Preview → Optional Google Calendar creation
```

Important safety principle:

> Always preview actions before allowing an AI workflow to touch real tools.

Calendar creation is disabled by default so the notebook can be safely run during workshops, demos, and GitHub review.

## 28. Install Google Calendar dependencies

This cell installs the Google API client libraries needed for Calendar integration.

In Colab, you can run this cell as-is. It is safe to rerun.

In [23]:
# Google Calendar API dependencies.
# These are only needed for Milestone 3.
!pip install google-api-python-client google-auth google-auth-oauthlib google-auth-httplib2 -q

print("Google Calendar dependencies installed.")

Google Calendar dependencies installed.


## 29. Calendar integration configuration

Keep `ENABLE_CALENDAR_CREATION = False` while previewing the workflow.

Only change it to `True` when you are ready to create real events on your Google Calendar.

In [24]:
# Milestone 3 configuration.
# Safe default: preview only. No real calendar events are created.
ENABLE_CALENDAR_CREATION = False

# Use "primary" to create events on your main Google Calendar.
# Advanced users can replace this with a specific calendar ID.
GOOGLE_CALENDAR_ID = "primary"

# Prefix makes test events easy to find and delete later.
CALENDAR_EVENT_PREFIX = "FocusFlow"

# Safety limit to prevent accidental event spam.
MAX_CALENDAR_EVENTS_TO_CREATE = 10

print(f"Calendar creation enabled: {ENABLE_CALENDAR_CREATION}")
print(f"Calendar ID: {GOOGLE_CALENDAR_ID}")
print(f"Event prefix: {CALENDAR_EVENT_PREFIX}")
print(f"Max events allowed: {MAX_CALENDAR_EVENTS_TO_CREATE}")

Calendar creation enabled: False
Calendar ID: primary
Event prefix: FocusFlow
Max events allowed: 10


## 30. Google Calendar setup for live event creation

The notebook is safe by default. It previews Calendar event payloads without touching your real calendar.

To create real Google Calendar events, you must complete a one-time Google Cloud OAuth setup and upload a local `credentials.json` file to Colab.

### What you need

- A Google account with Google Calendar access
- A Google Cloud project
- Google Calendar API enabled
- OAuth consent configured in **Testing** mode
- Your Google account added as a test user
- A Desktop OAuth client JSON file downloaded as `credentials.json`

### One-time Google Cloud setup

1. Go to Google Cloud Console.
2. Create or select a project, for example `FocusFlow Calendar Demo`.
3. Go to **APIs & Services → Library**.
4. Search for **Google Calendar API** and click **Enable**.
5. Go to **Google Auth Platform** or **OAuth consent screen**.
6. Configure the app:
   - App name: `FocusFlow Calendar Demo`
   - User support email: your email
   - Audience/user type: `External`
   - Publishing status: keep it in `Testing`
7. Go to **Data Access** and add this scope:

```text
https://www.googleapis.com/auth/calendar.events
```

8. Go to **Audience → Test users** and add the same Gmail account you will use during the Colab sign-in flow.
9. Go to **Clients** or **APIs & Services → Credentials**.
10. Create an OAuth client:
    - Application type: `Desktop app`
    - Name: `FocusFlow Colab Calendar Client`
11. Download the JSON file and rename it exactly:

```text
credentials.json
```

### Colab setup

1. In Colab, open the left file panel.
2. Upload `credentials.json`.
3. If Colab uploads it outside `/content`, the auth cell will try to copy it into the current working directory.
4. Do **not** upload `credentials.json` or `token.json` to GitHub.
5. Keep these in `.gitignore`:

```gitignore
credentials.json
token.json
*.json
```

### Running live calendar creation

Only after previewing and validating the calendar payloads, change:

```python
ENABLE_CALENDAR_CREATION = True
```

Then run the authentication cell.

The auth cell will print a Google authorization URL. Open it, approve access, and Google may redirect to a `localhost` page that fails to load. That is expected in Colab.

Copy the **full localhost URL** from the browser address bar and paste it back into Colab when prompted.

The URL should start with something like:

```text
http://localhost:8080/?state=...&code=...
```

After authentication completes, run the event creation cell once. Do not rerun it repeatedly unless you want duplicate calendar holds.

### Safety note

This notebook is intended for a controlled demo or workshop. Production apps should use a proper deployed OAuth redirect flow and should not rely on the local/manual Colab redirect workaround.


## 31. Build Google Calendar event payloads

Google Calendar expects event objects with fields such as:

```python
summary
start.dateTime
end.dateTime
```

This cell converts FocusFlow's calendar preview into Google Calendar event payloads.

We use `plan["calendar_preview"]` as the primary source because it represents the blocks that should actually become calendar holds.

In [25]:
def normalize_event_summary(summary, prefix=CALENDAR_EVENT_PREFIX):
    """Return a clean Google Calendar event title with the desired prefix."""
    summary = (summary or "Untitled FocusFlow Hold").strip()

    # Avoid double-prefixing if the LLM already included "FocusFlow:".
    if summary.lower().startswith(prefix.lower() + ":"):
        return summary

    return f"{prefix}: {summary}"


def parse_iso_datetime(value):
    """Parse an ISO 8601 datetime string for validation."""
    if not isinstance(value, str) or not value.strip():
        raise ValueError("Expected a non-empty ISO datetime string.")

    # Accept a trailing Z by converting it to +00:00.
    cleaned = value.strip().replace("Z", "+00:00")
    return datetime.fromisoformat(cleaned)


def build_calendar_event_payloads(plan, timezone=TIMEZONE):
    """Convert FocusFlow calendar preview items into Google Calendar event payloads."""
    if "calendar_preview" not in plan:
        raise KeyError("Plan is missing the 'calendar_preview' key.")

    preview_items = plan.get("calendar_preview", [])

    if not preview_items:
        raise ValueError("No calendar preview items found. Generate a FocusFlow plan first.")

    calendar_events = []

    for idx, item in enumerate(preview_items, start=1):
        summary = normalize_event_summary(item.get("summary"))
        start_time = item.get("start")
        end_time = item.get("end")

        if not start_time or not end_time:
            raise ValueError(f"Calendar preview item {idx} is missing start or end time: {item}")

        start_dt = parse_iso_datetime(start_time)
        end_dt = parse_iso_datetime(end_time)

        if end_dt <= start_dt:
            raise ValueError(
                f"Calendar preview item {idx} has end time before or equal to start time: {item}"
            )

        event = {
            "summary": summary,
            "description": (
                "Created by FocusFlow Agent.\n\n"
                "This hold was generated from a messy task dump and reviewed in preview mode before creation."
            ),
            "start": {
                "dateTime": start_time,
                "timeZone": timezone,
            },
            "end": {
                "dateTime": end_time,
                "timeZone": timezone,
            },
        }

        calendar_events.append(event)

    return calendar_events


calendar_event_payloads = build_calendar_event_payloads(plan, TIMEZONE)

print(f"Prepared {len(calendar_event_payloads)} Google Calendar event payloads.")
calendar_event_payloads[:2]

Prepared 4 Google Calendar event payloads.


[{'summary': 'FocusFlow: Finish project update',
  'description': 'Created by FocusFlow Agent.\n\nThis hold was generated from a messy task dump and reviewed in preview mode before creation.',
  'start': {'dateTime': '2026-05-21T09:00:00-07:00',
   'timeZone': 'America/Los_Angeles'},
  'end': {'dateTime': '2026-05-21T10:30:00-07:00',
   'timeZone': 'America/Los_Angeles'}},
 {'summary': 'FocusFlow: Prepare slide outline',
  'description': 'Created by FocusFlow Agent.\n\nThis hold was generated from a messy task dump and reviewed in preview mode before creation.',
  'start': {'dateTime': '2026-05-21T10:45:00-07:00',
   'timeZone': 'America/Los_Angeles'},
  'end': {'dateTime': '2026-05-21T12:00:00-07:00',
   'timeZone': 'America/Los_Angeles'}}]

## 32. Preview calendar holds before creating them

This table is the safety checkpoint.

Before creating real calendar events, verify that the titles, start times, end times, and timezone look correct.

In [26]:
calendar_payload_preview_rows = []

for event in calendar_event_payloads:
    calendar_payload_preview_rows.append({
        "summary": event["summary"],
        "start": event["start"]["dateTime"],
        "end": event["end"]["dateTime"],
        "timezone": event["start"]["timeZone"],
        "description": event["description"],
    })

calendar_payloads_df = pd.DataFrame(calendar_payload_preview_rows)
display(calendar_payloads_df)

,summary,start,end,timezone,description
0,FocusFlow: Finish project update,2026-05-21T09:00:00-07:00,2026-05-21T10:30:00-07:00,America/Los_Angeles,Created by FocusFlow Agent.\n\nThis hold was g...
1,FocusFlow: Prepare slide outline,2026-05-21T10:45:00-07:00,2026-05-21T12:00:00-07:00,America/Los_Angeles,Created by FocusFlow Agent.\n\nThis hold was g...
2,FocusFlow: Admin batch: pay rent + email Sam,2026-05-21T14:00:00-07:00,2026-05-21T14:30:00-07:00,America/Los_Angeles,Created by FocusFlow Agent.\n\nThis hold was g...
3,FocusFlow: Plan meetup agenda,2026-05-21T14:30:00-07:00,2026-05-21T15:15:00-07:00,America/Los_Angeles,Created by FocusFlow Agent.\n\nThis hold was g...


## 33. Validate calendar payloads

This cell performs basic validation before any real tool action:

- required fields exist
- start and end times are valid
- end time is after start time
- event count stays under the safety limit

This is the core engineering pattern for safe AI agents:

```text
Generate → Validate → Preview → Act
```

In [27]:
def validate_calendar_event_payloads(calendar_events, max_events=MAX_CALENDAR_EVENTS_TO_CREATE):
    """Validate Google Calendar event payloads before creation."""
    if not isinstance(calendar_events, list):
        raise TypeError("calendar_events must be a list.")

    if not calendar_events:
        raise ValueError("No calendar events to validate.")

    if len(calendar_events) > max_events:
        raise ValueError(
            f"Refusing to create {len(calendar_events)} events. "
            f"Safety limit is {max_events}."
        )

    for idx, event in enumerate(calendar_events, start=1):
        for key in ["summary", "start", "end"]:
            if key not in event:
                raise KeyError(f"Event {idx} is missing required key: {key}")

        start_time = event["start"].get("dateTime")
        end_time = event["end"].get("dateTime")

        start_dt = parse_iso_datetime(start_time)
        end_dt = parse_iso_datetime(end_time)

        if end_dt <= start_dt:
            raise ValueError(f"Event {idx} has invalid time range: {event}")

    print("Calendar payload validation passed.")
    print(f"Validated {len(calendar_events)} event payloads.")
    return True


validate_calendar_event_payloads(calendar_event_payloads)

Calendar payload validation passed.
Validated 4 event payloads.


True

## 34. Calendar creation safety check

This cell makes the notebook's behavior explicit before any real calendar action.

If `ENABLE_CALENDAR_CREATION` is `False`, the notebook stays in preview mode.

If `ENABLE_CALENDAR_CREATION` is `True`, the next cells can authenticate and create real Google Calendar events.

In [28]:
print("Calendar Creation Safety Check")
print("------------------------------")

if ENABLE_CALENDAR_CREATION:
    print("Calendar creation is ENABLED.")
    print("Running the creation cell will create real Google Calendar events.")
    print("Do not run the creation cell repeatedly unless you want duplicate holds.")
else:
    print("Calendar creation is DISABLED.")
    print("Preview mode only. No Google Calendar events will be created.")

Calendar Creation Safety Check
------------------------------
Calendar creation is DISABLED.
Preview mode only. No Google Calendar events will be created.


## 35. Authenticate with Google Calendar

This cell authenticates with Google Calendar using your own OAuth client file.

It only runs when:

```python
ENABLE_CALENDAR_CREATION = True
```

For GitHub safety, this notebook does **not** include `credentials.json` or `token.json`. Upload `credentials.json` manually in Colab when you want to run real Calendar creation.


In [29]:
# Authenticate with Google Calendar using your own OAuth client.
#
# Why this approach:
# - Colab's default auth can fail for Calendar write scopes.
# - run_local_server() redirects to localhost, which is awkward in Colab.
# - This manual flow lets you approve access in your browser and paste the
#   redirected localhost URL back into Colab.
#
# Do not commit credentials.json or token.json to GitHub.

import os
import shutil

# Required for a local OAuth redirect URL like http://localhost:8080
# in this controlled Colab/manual development flow.
# Do not use this pattern in a production web app.
os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

CALENDAR_SCOPES = [
    "https://www.googleapis.com/auth/calendar.events"
]

TOKEN_FILE = "token.json"
CREDENTIALS_FILE = "credentials.json"


def ensure_credentials_file(credentials_file=CREDENTIALS_FILE):
    """Make sure credentials.json is available in the current Colab working directory."""
    if os.path.exists(credentials_file):
        return credentials_file

    # Some Colab uploads can appear at the root level. Copy it into /content if needed.
    root_candidate = f"/{credentials_file}"
    if os.path.exists(root_candidate):
        shutil.copy(root_candidate, credentials_file)
        print(f"Copied {root_candidate} to {credentials_file}")
        return credentials_file

    raise FileNotFoundError(
        "credentials.json not found. Upload the OAuth client JSON file to Colab first. "
        "Do not upload this file to GitHub."
    )


def authenticate_google_calendar_manual_colab():
    """Authenticate with Google Calendar in Colab using a manual OAuth redirect flow."""
    creds = None

    # Reuse token if it already exists and is valid.
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(
            TOKEN_FILE,
            CALENDAR_SCOPES,
        )

    # Refresh or create a new token.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            ensure_credentials_file()

            flow = InstalledAppFlow.from_client_secrets_file(
                CREDENTIALS_FILE,
                CALENDAR_SCOPES,
            )

            # Fixed localhost redirect URI for manual copy/paste flow.
            flow.redirect_uri = "http://localhost:8080/"

            auth_url, _ = flow.authorization_url(
                prompt="consent",
                access_type="offline",
                include_granted_scopes="true",
            )

            print("Open this URL in your browser and approve access:")
            print(auth_url)
            print()
            print("After approval, Google may redirect to a localhost page that fails to load.")
            print("That is expected in Colab.")
            print("Copy the FULL localhost URL from your browser address bar and paste it below.")
            print("It should start with: http://localhost:8080/?code=... or http://localhost:8080/?state=...")

            redirected_url = input("Paste the full redirected localhost URL here: ").strip()

            flow.fetch_token(
                authorization_response=redirected_url,
            )

            creds = flow.credentials

        # Save token for reuse in this Colab runtime.
        # This file is local to the runtime and should not be committed.
        with open(TOKEN_FILE, "w", encoding="utf-8") as token:
            token.write(creds.to_json())

    service = build(
        "calendar",
        "v3",
        credentials=creds,
    )

    return service


if ENABLE_CALENDAR_CREATION:
    calendar_service = authenticate_google_calendar_manual_colab()
    print("Google Calendar authentication complete.")
else:
    calendar_service = None
    print("Calendar creation is disabled. Skipping Google authentication.")


Calendar creation is disabled. Skipping Google authentication.


## 36. Create Google Calendar holds

This is the real integration step.

It only runs when `ENABLE_CALENDAR_CREATION = True`.

The function uses the validated event payloads and creates events on the selected Google Calendar.

In [30]:
def create_google_calendar_holds(calendar_service, calendar_events, calendar_id=GOOGLE_CALENDAR_ID):
    """Create Google Calendar events from prepared event payloads."""
    if calendar_service is None:
        raise ValueError("calendar_service is None. Authenticate before creating events.")

    validate_calendar_event_payloads(calendar_events)

    created_events = []

    for event in calendar_events:
        created_event = calendar_service.events().insert(
            calendarId=calendar_id,
            body=event,
        ).execute()

        created_events.append({
            "summary": created_event.get("summary"),
            "start": created_event.get("start", {}).get("dateTime"),
            "end": created_event.get("end", {}).get("dateTime"),
            "htmlLink": created_event.get("htmlLink"),
            "id": created_event.get("id"),
        })

    return created_events


if ENABLE_CALENDAR_CREATION:
    created_calendar_events = create_google_calendar_holds(
        calendar_service=calendar_service,
        calendar_events=calendar_event_payloads,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    print(f"Created {len(created_calendar_events)} Google Calendar holds.")
else:
    created_calendar_events = []
    print("Calendar creation is disabled. No Google Calendar events were created.")

Calendar creation is disabled. No Google Calendar events were created.


## 37. Display created calendar links

If calendar creation was enabled, this cell displays the created events and their Google Calendar links.

In preview mode, this cell simply confirms that no events were created.

In [31]:
if created_calendar_events:
    created_events_df = pd.DataFrame(created_calendar_events)
    display(created_events_df)

    print("\nOpen these links to view the created calendar holds:")
    for event in created_calendar_events:
        print(event.get("htmlLink"))
else:
    print("No created events to display. Preview mode completed successfully.")

No created events to display. Preview mode completed successfully.


## 38. Cleanup note

If you enable calendar creation and create test events, open Google Calendar and delete the test holds manually.

They should be easy to find because the event titles start with:

```text
FocusFlow:
```

A future version can add a cleanup cell that deletes events created by this notebook, but manual cleanup is safer for the first workshop version.

## 39. Milestone 3 complete

FocusFlow now demonstrates the full action-oriented agent workflow:

```text
Messy task dump
→ structured plan
→ Slack-ready daily summary
→ Google Calendar event payloads
→ optional real calendar holds
```

This is the key lesson:

> A practical AI agent should generate structured outputs, validate them, preview the intended action, and only then touch real tools.

## 40. Milestone 4: One-cell FocusFlow demo runner

Milestone 4 turns the notebook into a simple **one-cell demo workflow**.

Instead of running every planning, Slack, and Calendar cell manually, you can:

1. paste a messy task dump,
2. choose whether to send Slack / create Calendar holds,
3. run one short cell.

The wrapper below calls the same helper functions from the earlier milestones:

```text
messy task dump
→ generate FocusFlow plan
→ show task table and schedule
→ validate Slack and Calendar outputs
→ optionally send Slack message
→ optionally create Google Calendar holds
→ show created event links
```

Safe defaults are still used. Slack posting and Google Calendar creation are **off by default** so the GitHub notebook can run without secrets or side effects.

In [32]:
def display_focusflow_result(plan, calendar_events=None, created_events=None):
    """Display the full FocusFlow result in a clean notebook-friendly format."""
    display(Markdown("## FocusFlow Result"))

    display(Markdown("### 1. Prioritized Task Table"))
    tasks_df = pd.DataFrame(plan["tasks"])
    preferred_task_columns = [
        "task", "category", "priority", "effort", "urgency", "when", "estimated_minutes", "reason"
    ]
    tasks_df = tasks_df[[col for col in preferred_task_columns if col in tasks_df.columns]]
    display(tasks_df)

    display(Markdown("### 2. Today's Schedule"))
    schedule_df = pd.DataFrame(plan["schedule"])
    preferred_schedule_columns = ["title", "start_time", "end_time", "type", "reason"]
    schedule_df = schedule_df[[col for col in preferred_schedule_columns if col in schedule_df.columns]]
    display(schedule_df)

    display(Markdown("### 3. Risks / Plan Check"))
    if plan.get("risks"):
        for risk in plan["risks"]:
            display(Markdown(f"- ⚠️ {risk}"))
    else:
        display(Markdown("- ✅ No major risks returned by the planner."))

    display(Markdown("### 4. Next Best Action"))
    display(Markdown(f"**{plan['next_best_action']}**"))

    display(Markdown("### 5. Slack Message Preview"))
    print(plan["slack_message"])

    display(Markdown("### 6. Google Calendar Holds Preview"))
    if calendar_events:
        calendar_preview_rows = []
        for event in calendar_events:
            calendar_preview_rows.append({
                "summary": event["summary"],
                "start": event["start"]["dateTime"],
                "end": event["end"]["dateTime"],
                "timezone": event["start"].get("timeZone"),
            })
        display(pd.DataFrame(calendar_preview_rows))
    else:
        display(Markdown("No calendar event payloads were generated."))

    if created_events:
        display(Markdown("### 7. Created Google Calendar Event Links"))
        display(pd.DataFrame(created_events))
        for event in created_events:
            if event.get("htmlLink"):
                display(Markdown(f"- [{event.get('summary', 'Calendar event')}]({event['htmlLink']})"))


def run_focusflow_end_to_end(
    task_dump,
    today_date=TODAY_DATE,
    timezone=TIMEZONE,
    use_mock=USE_MOCK_MODE,
    send_to_slack=False,
    create_calendar_holds=False,
    slack_webhook_url=None,
    calendar_service=None,
    display_outputs=True,
):
    """Run FocusFlow end-to-end from one notebook cell.

    This is the main Milestone 4 wrapper.

    Args:
        task_dump: Messy natural-language task list.
        today_date: Date used for schedule generation.
        timezone: IANA timezone string.
        use_mock: If True, use deterministic mock output. If False, call the configured LLM.
        send_to_slack: If True, post the Slack message to a real Slack webhook.
        create_calendar_holds: If True, authenticate and create real Google Calendar events.
        slack_webhook_url: Optional Slack webhook URL. If omitted, the function checks env var and may prompt.
        calendar_service: Optional authenticated Google Calendar service.
        display_outputs: If True, display tables and previews.

    Returns:
        Dictionary containing the plan, calendar payloads, Slack status, and created calendar events.
    """
    if not task_dump or not task_dump.strip():
        raise ValueError("Please provide a non-empty task dump.")

    display(Markdown("# FocusFlow One-cell Run"))
    display(Markdown("Generating plan..."))

    # 1. Generate and validate plan.
    plan = generate_focusflow_plan(
        task_dump=task_dump,
        today_date=today_date,
        timezone=timezone,
        use_mock=use_mock,
    )

    if not validate_plan(plan, verbose=False):
        raise ValueError("FocusFlow plan failed validation. Check the model output or mock plan structure.")

    # 2. Build and validate Calendar payloads.
    calendar_events = build_calendar_event_payloads(plan, timezone=timezone)
    validate_calendar_event_payloads(calendar_events)

    # 3. Validate Slack message.
    slack_valid = validate_slack_message(plan)

    slack_status = "skipped"
    created_events = []

    # 4. Optional Slack action.
    if send_to_slack:
        webhook = (slack_webhook_url or os.environ.get("SLACK_WEBHOOK_URL", "")).strip()
        if not webhook:
            webhook = getpass("Enter your Slack webhook URL: ").strip()
        post_to_slack_result = post_to_slack(plan["slack_message"], webhook)
        slack_status = "posted" if post_to_slack_result else "failed"
    else:
        slack_status = "preview_only"

    # 5. Optional Calendar action.
    if create_calendar_holds:
        if calendar_service is None:
            display(Markdown("Authenticating with Google Calendar..."))
            calendar_service = authenticate_google_calendar_manual_colab()

        created_events = create_google_calendar_holds(
            calendar_service=calendar_service,
            calendar_events=calendar_events,
            calendar_id=GOOGLE_CALENDAR_ID,
        )
    else:
        created_events = []

    # 6. Display final output.
    if display_outputs:
        display_focusflow_result(plan, calendar_events=calendar_events, created_events=created_events)

        display(Markdown("### Run Summary"))
        display(Markdown(f"- Slack status: **{slack_status}**"))
        display(Markdown(f"- Calendar holds created: **{len(created_events)}**"))
        if not send_to_slack:
            display(Markdown("- Slack posting was disabled. The message was shown as a preview."))
        if not create_calendar_holds:
            display(Markdown("- Google Calendar creation was disabled. Calendar holds were shown as a preview."))

    return {
        "plan": plan,
        "calendar_events": calendar_events,
        "slack_valid": slack_valid,
        "slack_status": slack_status,
        "created_calendar_events": created_events,
    }


# Backward-compatible alias with a short name for demos.
run_focusflow = run_focusflow_end_to_end

print("Milestone 4 one-cell runner is ready.")

Milestone 4 one-cell runner is ready.


## 41. One-cell demo control panel

This is the cell to use during the live demo.

For a safe GitHub/demo run, leave both action flags as `False`:

```python
SEND_TO_SLACK = False
CREATE_CALENDAR_HOLDS = False
```

For a real live run after setup is complete:

```python
SEND_TO_SLACK = True
CREATE_CALENDAR_HOLDS = True
```

Before enabling real actions, make sure:

- your OpenAI API key is configured if `USE_MOCK_MODE = False`,
- your Slack webhook is available through `SLACK_WEBHOOK_URL` or entered when prompted,
- `credentials.json` is uploaded to Colab,
- Google Cloud OAuth is configured with the Calendar Events scope,
- your Gmail is added as a test user,
- you are comfortable creating real calendar events.

Do not commit `credentials.json`, `token.json`, `.env`, Slack webhooks, or API keys to GitHub.

In [33]:
# Milestone 4 demo cell.
# Paste the task dump, set the two action flags, and run this one cell.

TASK_DUMP_FOR_DEMO = """
I need to prepare slides for Friday, email Sam, review an AI paper,
book a dentist appointment, finish project update, go to the gym,
pay rent, and plan the meetup agenda.

I have 4 hours today and prefer deep work in the morning.
"""

# Safe defaults for GitHub and workshop dry runs.
# Change these to True only when you are ready to trigger real external actions.
SEND_TO_SLACK = False
CREATE_CALENDAR_HOLDS = False

focusflow_run = run_focusflow(
    task_dump=TASK_DUMP_FOR_DEMO,
    today_date=TODAY_DATE,
    timezone=TIMEZONE,
    use_mock=USE_MOCK_MODE,
    send_to_slack=SEND_TO_SLACK,
    create_calendar_holds=CREATE_CALENDAR_HOLDS,
)


# FocusFlow One-cell Run

Generating plan...

Calendar payload validation passed.
Validated 4 event payloads.
Slack message validation passed.
Message length: 434 characters


## FocusFlow Result

### 1. Prioritized Task Table

...

### Run Summary

- Slack status: **preview_only**

- Calendar holds created: **0**

- Slack posting was disabled. The message was shown as a preview.

- Google Calendar creation was disabled. Calendar holds were shown as a preview.

## 42. Milestone 4 complete

You now have a one-cell FocusFlow demo workflow.

The full notebook progression is:

```text
Milestone 1: Generate a structured FocusFlow plan
Milestone 2: Send the Slack summary
Milestone 3: Create Google Calendar holds
Milestone 4: Run the full workflow from one clean demo cell
```

This gives you a practical follow-along flow for a short session while still preserving safe preview defaults for GitHub.